In [54]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import networkx as nx

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


In [55]:
G = nx.DiGraph()

import random

accounts = [f"A{i}" for i in range(200)]

for _ in range(2000):
    sender = random.choice(accounts)
    receiver = random.choice(accounts)
    if sender != receiver:
        amount = random.randint(1000, 50000)
        G.add_edge(sender, receiver, amount=amount)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())




Nodes: 200
Edges: 1930


In [57]:
node_list = list(G.nodes())
node_to_idx = {node: idx for idx, node in enumerate(node_list)}

print("Total nodes:", len(node_list))



Total nodes: 200


In [ ]:
features = []
labels = []

for node in node_list:
    in_deg = G.in_degree(node)
    out_deg = G.out_degree(node)

    total_in = sum(G[u][node].get('amount', 0) for u in G.predecessors(node)) if in_deg > 0 else 0
    total_out = sum(G[node][v].get('amount', 0) for v in G.successors(node)) if out_deg > 0 else 0

    fan_in_ratio = in_deg / (in_deg + out_deg + 1)
    amount_ratio = total_in / (total_in + total_out + 1)

    features.append([
        in_deg,
        out_deg,
        total_in,
        total_out,
        fan_in_ratio,
        amount_ratio
    ])

    if in_deg > 5 and out_deg > 5:
        labels.append(1)
    else:
        labels.append(0)

features = np.array(features)
labels = np.array(labels)

print("Normal:", sum(labels == 0))
print("Fraud:", sum(labels == 1))



Normal: 0
Fraud: 1


In [66]:
scaler = StandardScaler()
features = scaler.fit_transform(features)

x = torch.tensor(features, dtype=torch.float)
y = torch.tensor(labels, dtype=torch.long)

edges = []
for u, v in G.edges():
    edges.append([node_to_idx[u], node_to_idx[v]])

edge_index = torch.tensor(np.array(edges).T, dtype=torch.long)

data = Data(x=x, edge_index=edge_index, y=y)

print("Feature shape:", data.x.shape)



Feature shape: torch.Size([200, 6])


In [69]:
scaler = StandardScaler()
features = scaler.fit_transform(features)

x = torch.tensor(features, dtype=torch.float)
y = torch.tensor(labels, dtype=torch.long)

edges = []
for u, v in G.edges():
    edges.append([node_to_idx[u], node_to_idx[v]])

edge_index = torch.tensor(np.array(edges).T, dtype=torch.long)

data = Data(x=x, edge_index=edge_index, y=y)

print("Feature shape:", data.x.shape)


Feature shape: torch.Size([200, 6])


In [61]:
class GraphSAGE(nn.Module):
    def __init__(self, in_channels):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(in_channels, 64)
        self.conv2 = SAGEConv(64, 32)
        self.lin = nn.Linear(32, 2)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.lin(x)
        return x



In [62]:
model = GraphSAGE(in_channels=6)

optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

fraud_count = (data.y == 1).sum().item()
normal_count = (data.y == 0).sum().item()

# Safety: avoid division by zero
if fraud_count == 0:
    weight_fraud = 1.0
else:
    weight_fraud = (normal_count / fraud_count) * 0.5   # smoother correction

weight_normal = 1.0

weights = torch.tensor([weight_normal, weight_fraud], dtype=torch.float)

criterion = nn.CrossEntropyLoss(weight=weights)

print("Normal:", normal_count)
print("Fraud:", fraud_count)
print("Fraud weight:", round(weight_fraud, 3))


Normal: 33
Fraud: 167
Fraud weight: 0.099


In [63]:
best_f1 = 0

for epoch in range(1, 301):

    loss = train()
    acc, f1, precision, recall = test()

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "gnn_model.pth")

    if epoch % 20 == 0:
        print(f"Epoch {epoch}")
        print(f"Loss: {loss:.4f}")
        print(f"Accuracy: {acc:.4f}")
        print(f"F1: {f1:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print("-" * 30)

print("Best F1 achieved:", round(best_f1, 4))



c:\Users\Neranjana\Desktop\UPI_ANALYST_CONSOLE\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Neranjana\Desktop\UPI_ANALYST_CONSOLE\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Neranjana\Desktop\UPI_ANALYST_CONSOLE\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

Epoch 20
Loss: 0.3217
Accuracy: 0.8750
F1: 0.9180
Precision: 1.0000
Recall: 0.8485
------------------------------
Epoch 40
Loss: 0.1140
Accuracy: 0.9750
F1: 0.9846
Precision: 1.0000
Recall: 0.9697
------------------------------
Epoch 60
Loss: 0.0448
Accuracy: 1.0000
F1: 1.0000
Precision: 1.0000
Recall: 1.0000
------------------------------
Epoch 80
Loss: 0.0184
Accuracy: 1.0000
F1: 1.0000
Precision: 1.0000
Recall: 1.0000
------------------------------
Epoch 100
Loss: 0.0093
Accuracy: 1.0000
F1: 1.0000
Precision: 1.0000
Recall: 1.0000
------------------------------
Epoch 120
Loss: 0.0055
Accuracy: 1.0000
F1: 1.0000
Precision: 1.0000
Recall: 1.0000
------------------------------
Epoch 140
Loss: 0.0036
Accuracy: 1.0000
F1: 1.0000
Precision: 1.0000
Recall: 1.0000
------------------------------
Epoch 160
Loss: 0.0025
Accuracy: 1.0000
F1: 1.0000
Precision: 1.0000
Recall: 1.0000
------------------------------
Epoch 180
Loss: 0.0019
Accuracy: 1.0000
F1: 1.0000
Precision: 1.0000
Recall: 1.0000


In [ ]:
best_f1 = 0

for epoch in range(1, 301):

    loss = train()
    acc, f1, precision, recall = test()

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "gnn_model.pth")

    if epoch % 20 == 0:
        print(f"Epoch {epoch}")
        print(f"Loss: {loss:.4f}")
        print(f"Accuracy: {acc:.4f}")
        print(f"F1: {f1:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print("-" * 30)

print("Best F1 achieved:", round(best_f1, 4))


In [70]:
torch.save(model.state_dict(), "gnn_model.pth")
print("Model saved successfully.")


Model saved successfully.
